In [ ]:
'''
새로운 코드를 작성해줘. filtered_concat_translated_problems.csv에서 generated_question 열에 codeforce라는 문자열이 포함된 문제(소문자로 변환한 다음에 검사)만 추출해서 따로 codeforce_problems.csv 로 저장하게 해줘. 그리고 problem_testcases.csv를 열어서(파일 크기가 크니 배치 10000 단위로) 해당되는 problem_id들을 찾아서 추출한 다음 codeforce_testcases.csv로 저장하게 해줘. 이해 안 되는 부분은 말해주고

In [1]:
import os
import pandas as pd
import time

# ---------------------------------------------------------
# [설정값 및 파일 경로]
# ---------------------------------------------------------
PROBLEMS_FILE = 'filtered_concat_translated_problems.csv'
TESTCASES_FILE = 'problem_testcases.csv'

OUTPUT_PROB_FILE = 'codeforce_problems.csv'
OUTPUT_TC_FILE = 'codeforce_testcases.csv'

CHUNK_SIZE = 10000

print("🚀 [Step 1] Codeforces 문제 필터링 시작...")
start_time = time.time()

# ---------------------------------------------------------
# 1. 문제(Problems) 필터링
# ---------------------------------------------------------
try:
    df_problems = pd.read_csv(PROBLEMS_FILE)
    
    # NaN 값 방어를 위해 먼저 문자열로 변환하고 소문자로 만든 뒤 'codeforce' 포함 여부 검사
    mask = df_problems['generated_question'].astype(str).str.lower().str.contains('codeforce')
    
    # 조건에 맞는 행만 추출
    df_cf_problems = df_problems[mask].copy()
    
    # 추출된 문제들의 id 목록을 집합(Set)으로 저장해둠 (테스트케이스 필터링 시 속도 최적화)
    # id 열의 데이터 타입 불일치를 막기 위해 모두 문자열로 통일
    cf_problem_ids = set(df_cf_problems['id'].astype(str).str.replace(r'\.0$', '', regex=True))
    
    print(f"  - 전체 {len(df_problems)}문제 중 'codeforce'가 포함된 문제 {len(df_cf_problems)}개 발견!")
    
    if len(df_cf_problems) == 0:
        print("  ⚠️ Codeforces 문제가 하나도 없습니다. 작업을 종료합니다.")
        sys.exit(0)
        
    # 필터링된 문제들을 새로운 CSV로 저장
    df_cf_problems.to_csv(OUTPUT_PROB_FILE, index=False, encoding='utf-8-sig')
    print(f"  ✔ {OUTPUT_PROB_FILE} 저장 완료.")

except Exception as e:
    print(f"❌ 문제 파일 처리 중 에러 발생: {e}")
    sys.exit(1)


print("\n🚀 [Step 2] 해당 문제들의 테스트케이스 추출 시작 (대용량 청크 처리)...")

# ---------------------------------------------------------
# 2. 테스트케이스(Testcases) 청크 필터링
# ---------------------------------------------------------
is_first_chunk = True
total_extracted_tcs = 0

try:
    chunk_iter = pd.read_csv(TESTCASES_FILE, chunksize=CHUNK_SIZE)
    
    for chunk_idx, df_tc_chunk in enumerate(chunk_iter, start=1):
        # 매칭 오류를 막기 위해 problem_id를 순수 문자열로 변환
        chunk_prob_ids = df_tc_chunk['problem_id'].astype(str).str.replace(r'\.0$', '', regex=True)
        
        # Step 1에서 찾아낸 Codeforces 문제 ID 목록에 포함되는 행만 싹 골라내기
        filtered_tc_chunk = df_tc_chunk[chunk_prob_ids.isin(cf_problem_ids)]
        
        if not filtered_tc_chunk.empty:
            # 파일이 처음 생성될 때만 헤더(열 이름)를 넣고, 이후에는 이어붙이기(append)
            mode = 'w' if is_first_chunk else 'a'
            header = is_first_chunk
            
            filtered_tc_chunk.to_csv(OUTPUT_TC_FILE, mode=mode, header=header, index=False, encoding='utf-8-sig')
            
            is_first_chunk = False
            total_extracted_tcs += len(filtered_tc_chunk)
            
        # 진행 상황 출력 (10만 개 단위마다 출력하여 터미널 도배 방지)
        if (chunk_idx * CHUNK_SIZE) % 100000 == 0:
            print(f"  - {chunk_idx * CHUNK_SIZE:,}개 행 스캔 완료... (현재까지 {total_extracted_tcs:,}개 테스트케이스 추출됨)")

except Exception as e:
    print(f"❌ 테스트케이스 파일 처리 중 에러 발생: {e}")
    sys.exit(1)

elapsed_time = time.time() - start_time

print("\n🎉 모든 분리 추출 작업이 성공적으로 완료되었습니다!")
print(f"⏱️ 총 소요 시간: {elapsed_time:.1f}초")
print(f"📁 결과물 1: {OUTPUT_PROB_FILE} (문제 {len(df_cf_problems):,}개)")
print(f"📁 결과물 2: {OUTPUT_TC_FILE} (테스트케이스 {total_extracted_tcs:,}개)")

🚀 [Step 1] Codeforces 문제 필터링 시작...
  - 전체 3544문제 중 'codeforce'가 포함된 문제 5개 발견!
  ✔ codeforce_problems.csv 저장 완료.

🚀 [Step 2] 해당 문제들의 테스트케이스 추출 시작 (대용량 청크 처리)...
  - 100,000개 행 스캔 완료... (현재까지 0개 테스트케이스 추출됨)
  - 200,000개 행 스캔 완료... (현재까지 0개 테스트케이스 추출됨)
  - 300,000개 행 스캔 완료... (현재까지 167개 테스트케이스 추출됨)
  - 400,000개 행 스캔 완료... (현재까지 167개 테스트케이스 추출됨)
  - 500,000개 행 스캔 완료... (현재까지 167개 테스트케이스 추출됨)
  - 600,000개 행 스캔 완료... (현재까지 167개 테스트케이스 추출됨)
  - 700,000개 행 스캔 완료... (현재까지 549개 테스트케이스 추출됨)
  - 800,000개 행 스캔 완료... (현재까지 549개 테스트케이스 추출됨)
  - 900,000개 행 스캔 완료... (현재까지 549개 테스트케이스 추출됨)
  - 1,000,000개 행 스캔 완료... (현재까지 549개 테스트케이스 추출됨)
  - 1,100,000개 행 스캔 완료... (현재까지 736개 테스트케이스 추출됨)
  - 1,200,000개 행 스캔 완료... (현재까지 871개 테스트케이스 추출됨)
  - 1,300,000개 행 스캔 완료... (현재까지 871개 테스트케이스 추출됨)

🎉 모든 분리 추출 작업이 성공적으로 완료되었습니다!
⏱️ 총 소요 시간: 72.0초
📁 결과물 1: codeforce_problems.csv (문제 5개)
📁 결과물 2: codeforce_testcases.csv (테스트케이스 871개)


In [13]:
import pandas as pd

def apply_perfect_ssajun_cipher():
    print("🚀 [PJT] 싸준 테마 변환(1:1 정밀 치환) 작업을 시작합니다...")

    try:
        p_df = pd.read_csv('codeforce_problems (전).csv')
        t_df = pd.read_csv('problem_testcases.csv')
    except FileNotFoundError:
        print("❌ 원본 CSV 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 1. 문제 설명용 텍스트 매핑 (스토리/예시 변환)
    story_replacements = [
        ('"for"는 "codeforces", "for", "therefore"의 부분 문자열이지만', '"jun"은 "ssajunpjt", "jun", "juniors"의 부분 문자열이지만'),
        ('"for"는', '"jun"은'),
        ('"therefore"', '"juniors"'),
        ('Codeforces', 'SsajunProj'),
        ('codef0rces', 'ssajunpr0j'),
        ('dpedepqbft', 'ttbkvoufbl'),
        ('secrofedoc', 'jorpnujass'),
        ('orcesfedoc', 'nprojujass'),
        ('rocesfedoc', 'pnrojujass'),
        ('codeforces', 'sajunapsur') # 12767번 문제 설명을 위한 기본 단어 교체
    ]

    # --- 1) Problems 데이터 변환 ---
    text_columns = ['generated_question', 'generated_exp_input', 'generated_exp_output', 'generated_exp_example']
    for col in text_columns:
        if col in p_df.columns:
            for old_str, new_str in story_replacements:
                p_df[col] = p_df[col].astype(str).str.replace(old_str, new_str, regex=False)
            
            # 특정 문제 예외 처리 덮어쓰기 (길이/암호화 로직 특수 문제들)
            p_df.loc[p_df['id'] == 4424, col] = p_df.loc[p_df['id'] == 4424, col].str.replace('sajunapsur', 'ssajunteam', regex=False)
            p_df.loc[p_df['id'] == 22051, col] = p_df.loc[p_df['id'] == 22051, col].str.replace('sajunapsur', 'ssajunproj', regex=False)

    # --- 2) Testcases 데이터 변환 ---
    t_df['input'] = t_df['input'].astype(str)
    t_df['output'] = t_df['output'].astype(str)

    # 발굴해낸 21개의 의심 단어 1:1 완벽 치환 딕셔너리
    cipher_dict = {
        'cndeforces': 'sljunapsur', 'coddforces': 'sajjnapsur', 'codeeorcfs': 'sajuuapsnr',
        'codefnrces': 'sajunlpsur', 'codefoqces': 'sajunaosur', 'codeforcer': 'sajunapsup',
        'codeforces': 'sajunapsur', 'codeforcfs': 'sajunapsnr', 'codeforcse': 'sajunapsru',
        'codeforecq': 'sajunapuso', 'codeforecr': 'sajunapusp', 'codeforrec': 'sajunappus',
        'codefoscer': 'sajunarsup', 'codefprces': 'sajunmpsur', 'codefsecor': 'sajunrusap',
        'codegorces': 'sajudapsur', 'codegorecq': 'sajudapuso', 'coderofces': 'sajupansur',
        'codfforcer': 'sajnnapsup', 'codfforces': 'sajnnapsur', 'coeeforces': 'sauunapsur'
    }

    # 12767번 등 테스트케이스 내의 codeforce 변형 단어들을 일괄 치환
    for old_word, new_word in cipher_dict.items():
        # 단어 단위로 정확히 치환하기 위해 정규표현식(\b) 사용
        t_df['input'] = t_df['input'].str.replace(rf'\b{old_word}\b', new_word, regex=True)

    # 22051번 특수 암호화 문제 테스트케이스 처리
    mask_22051 = t_df['problem_id'] == 22051
    t_df.loc[mask_22051, 'input'] = t_df.loc[mask_22051, 'input'].str.replace('rocesfedoc', 'pnrojujass', regex=False)
    t_df.loc[mask_22051, 'output'] = t_df.loc[mask_22051, 'output'].str.replace('sajunapsur', 'ssajunproj', regex=False)

    # 3. 새로운 파일로 저장
    p_df.to_csv('ssajun_problems.csv', index=False, encoding='utf-8-sig')
    t_df.to_csv('ssajun_testcases.csv', index=False, encoding='utf-8-sig')
    
    print("✅ 변환 완료! 모든 논리가 보존된 'ssajun_problems.csv'와 'ssajun_testcases.csv'가 성공적으로 생성되었습니다.")

# 함수 실행
apply_perfect_ssajun_cipher()

🚀 [PJT] 싸준 테마 변환(1:1 정밀 치환) 작업을 시작합니다...


KeyboardInterrupt: 

In [12]:
import pandas as pd
import subprocess
import tempfile
import os

def verify_all_solutions(problems_file='ssajun_problems.csv', testcases_file='ssajun_testcases.csv'):
    print("🚀 전체 문제의 솔루션 코드 검증을 시작합니다...\n")

    #pd.set_option('display.max_rows',None)
    
    # 1. 데이터 불러오기
    try:
        p_df = pd.read_csv(problems_file)
        t_df = pd.read_csv(testcases_file)
    except FileNotFoundError:
        print("❌ CSV 파일을 찾을 수 없습니다. 파일명을 확인해 주세요.")
        return

    all_results = []
    total_problems = len(p_df)
    
    # 2. 모든 문제를 하나씩 순회 (For 문)
    for idx, problem_row in p_df.iterrows():
        problem_id = problem_row['id']
        
        print(f"==================================================")
        print(f"📝 [문제 ID: {problem_id}] 검증 진행 중... ({idx+1}/{total_problems})")
        
        # 솔루션 코드가 없는 경우 방어 로직
        if pd.isna(problem_row['solution']):
            print(f"❌ 문제 ID {problem_id}번의 솔루션 코드가 존재하지 않습니다. 스킵합니다.\n")
            continue
            
        solution_code = problem_row['solution']
        
        # 해당 문제의 테스트케이스 추출
        target_testcases = t_df[t_df['problem_id'] == problem_id].copy()
        if target_testcases.empty:
            print(f"❌ 문제 ID {problem_id}번의 테스트케이스가 존재하지 않습니다. 스킵합니다.\n")
            continue

        # subprocess로 안전하게 실행하기 위해 임시 파일 생성
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as temp_py:
            temp_py.write(solution_code)
            temp_py_path = temp_py.name

        print(f"✅ 총 {len(target_testcases)}개의 테스트케이스를 검사합니다.")

        problem_results = []
        match_count = 0
        
        # 3. 테스트케이스 순회하며 코드 실행
        for tc_index, row in target_testcases.iterrows():
            tc_input = str(row['input'])
            expected_output = str(row['output']).strip()
            
            actual_output = ""
            status = "PASS"
            
            try:
                # 💡 타임아웃 5초로 변경
                process = subprocess.run(
                    ['python', temp_py_path],
                    input=tc_input,
                    text=True,
                    capture_output=True,
                    timeout=5  # <--- 요청하신 5초 타임아웃 적용 부분
                )
                
                # 에러 발생 시
                if process.returncode != 0:
                    actual_output = process.stderr.strip()
                    status = "ERROR"
                else:
                    actual_output = process.stdout.strip()
                    # 정답 비교
                    if actual_output == expected_output:
                        status = "MATCH"
                        match_count += 1
                    else:
                        status = "MISMATCH"
                        
            except subprocess.TimeoutExpired:
                actual_output = "시간 초과 (Timeout: 5초)"
                status = "TIMEOUT"
            except Exception as e:
                actual_output = f"실행 오류: {str(e)}"
                status = "ERROR"

            # 개별 테스트케이스 결과 기록
            result_dict = {
                'Problem ID': problem_id,
                'Testcase Order': row.get('testcase_order', tc_index + 1),
                'Input': tc_input,
                'Expected Output': expected_output,
                'Actual Output': actual_output,
                'Result': status
            }
            problem_results.append(result_dict)
            all_results.append(result_dict)

        # 임시 파이썬 파일 삭제 (정리)
        if os.path.exists(temp_py_path):
            os.remove(temp_py_path)

        # 문제별 최종 통계 출력
        print(f"\n📊 통계: {match_count} / {len(target_testcases)} 통과\n")
        
        # 4. 문제별 결과를 테이블(데이터프레임)로 즉시 표시
        problem_result_df = pd.DataFrame(problem_results)
        try:
            from IPython.display import display
            display(problem_result_df)  # Jupyter 환경에서 예쁜 표로 출력
        except ImportError:
            print(problem_result_df.to_string())
            
        print("\n")

    # 전체 검증 완료
    print("==================================================")
    print(f"🎉 모든 문제(총 {total_problems}개)의 검증 및 출력이 완료되었습니다!")
    
    # 전체 기록이 담긴 데이터프레임 반환 (필요시 to_csv로 저장 가능)
    final_all_results_df = pd.DataFrame(all_results)
    return final_all_results_df

# 함수 실행 (반환된 전체 결과 테이블을 변수에 저장)
all_results_table = verify_all_solutions()

🚀 전체 문제의 솔루션 코드 검증을 시작합니다...

📝 [문제 ID: 4424.0] 검증 진행 중... (1/5)
✅ 총 167개의 테스트케이스를 검사합니다.

📊 통계: 167 / 167 통과



,Problem ID,Testcase Order,Input,Expected Output,Actual Output,Result
0,4424.0,1,5\n6\nabccba\n2\ncf\n4\nadfa\n8\nabaazaba\n2\n...,YES\nNO\nYES\nNO\nNO,YES\nNO\nYES\nNO\nNO,MATCH
1,4424.0,2,1\n2\nay\n,NO,NO,MATCH
2,4424.0,3,1\n2\nae\n,NO,NO,MATCH
3,4424.0,4,1\n2\nya\n,NO,NO,MATCH
4,4424.0,5,1\n2\nzb\n,NO,NO,MATCH
5,4424.0,6,1\n4\nzddb\n,NO,NO,MATCH
6,4424.0,7,1\n2\nbz\n,NO,NO,MATCH
7,4424.0,8,1\n2\nab\n,NO,NO,MATCH
8,4424.0,9,3\n2\nbz\n2\nyz\n2\nab\n,NO\nNO\nNO,NO\nNO\nNO,MATCH
9,4424.0,10,1\n4\nauuy\n,NO,NO,MATCH




📝 [문제 ID: 12515.0] 검증 진행 중... (2/5)
✅ 총 180개의 테스트케이스를 검사합니다.

📊 통계: 180 / 180 통과



,Problem ID,Testcase Order,Input,Expected Output,Actual Output,Result
0,12515.0,1,1_wat\n2\n2_wat\nwat_1\n,Yes,Yes,MATCH
1,12515.0,2,000\n3\n00\nooA\noOo\n,No,No,MATCH
2,12515.0,3,_i_\n3\n__i_\n_1_\nI\n,No,No,MATCH
3,12515.0,4,La0\n3\n2a0\nLa1\n1a0\n,No,No,MATCH
4,12515.0,5,abc\n1\naBc\n,No,No,MATCH
5,12515.0,6,0Lil\n2\nLIL0\n0Ril\n,Yes,Yes,MATCH
6,12515.0,7,iloO\n3\niIl0\noIl0\nIooO\n,Yes,Yes,MATCH
7,12515.0,8,L1il0o1L1\n5\niLLoLL\noOI1Io10il\nIoLLoO\nO01i...,Yes,Yes,MATCH
8,12515.0,9,ELioO1lOoOIOiLoooi1iolul1O\n7\nOoEIuOIl1ui1010...,Yes,Yes,MATCH
9,12515.0,10,0blo7X\n20\n1oobb6\nXIXIO2X\n2iYI2\n607XXol\n2...,Yes,Yes,MATCH




📝 [문제 ID: 12767.0] 검증 진행 중... (3/5)
✅ 총 202개의 테스트케이스를 검사합니다.

📊 통계: 202 / 202 통과



...



📝 [문제 ID: 19990.0] 검증 진행 중... (4/5)
✅ 총 187개의 테스트케이스를 검사합니다.

📊 통계: 187 / 187 통과



,Problem ID,Testcase Order,Input,Expected Output,Actual Output,Result
0,19990.0,1,5\na\naba\nabacaba\nba\naba\n,YES\na\nba\naba\naba\nabacaba,YES\na\nba\naba\naba\nabacaba,MATCH
1,19990.0,2,5\na\nabacaba\nba\naba\nabab\n,NO,NO,MATCH
2,19990.0,3,3\nqwerty\nqwerty\nqwerty\n,YES\nqwerty\nqwerty\nqwerty,YES\nqwerty\nqwerty\nqwerty,MATCH
3,19990.0,4,1\nwronganswer\n,YES\nwronganswer,YES\nwronganswer,MATCH
4,19990.0,5,3\na\nb\nab\n,NO,NO,MATCH
5,19990.0,6,2\nababaab\nabaab\n,YES\nabaab\nababaab,YES\nabaab\nababaab,MATCH
6,19990.0,7,2\nq\nqq\n,YES\nq\nqq,YES\nq\nqq,MATCH
7,19990.0,8,5\nabab\nbab\nba\nab\na\n,NO,NO,MATCH
8,19990.0,9,3\nb\nc\nd\n,NO,NO,MATCH
9,19990.0,10,3\naba\nbab\nababa\n,NO,NO,MATCH




📝 [문제 ID: 22051.0] 검증 진행 중... (5/5)
✅ 총 135개의 테스트케이스를 검사합니다.

📊 통계: 135 / 135 통과



,Problem ID,Testcase Order,Input,Expected Output,Actual Output,Result
0,22051.0,1,10\npnrojujass\n,ssajunproj,ssajunproj,MATCH
1,22051.0,2,16\nplmaetwoxesisiht\n,thisisexampletwo,thisisexampletwo,MATCH
2,22051.0,3,1\nz\n,z,z,MATCH
3,22051.0,4,2\nir\n,ri,ri,MATCH
4,22051.0,5,3\nilj\n,jli,jli,MATCH
5,22051.0,6,4\njfyy\n,yyjf,yyjf,MATCH
6,22051.0,7,6\nkrdych\n,hcyrkd,hcyrkd,MATCH
7,22051.0,8,60\nfnebsopcvmlaoecpzmakqigyuutueuozjxutlwwioc...,jqprcbfgsxwgjhmkehcoiwwltuxjzokamzpalobnfespcv...,jqprcbfgsxwgjhmkehcoiwwltuxjzokamzpalobnfespcv...,MATCH
8,22051.0,9,64\nhnlzzhrvqnldswxfsrowfhmyzbxtyoxhogudasgywx...,ywnvcnclsibreesigzhycyxwygsadugofxwsdlnqzlhnzh...,ywnvcnclsibreesigzhycyxwygsadugofxwsdlnqzlhnzh...,MATCH
9,22051.0,10,97\nqnqrmdhmbubaijtwsecbidqouhlecladwgwcuxbigc...,crgjulypijzhyypkqtxbibuoxhkagycauuqgchyaokulsb...,crgjulypijzhyypkqtxbibuoxhkagycauuqgchyaokulsb...,MATCH




🎉 모든 문제(총 5개)의 검증 및 출력이 완료되었습니다!
